# Transformer Block

Now we have attention. But attention alone is not enough. We need
to wrap it in a complete building block that the model can stack
many times. This is the transformer block.

GPT-2 Small stacks 12 of these blocks. GPT-3 stacks 96 of them.
LLaMA 70B stacks 80. Every modern language model is made by
repeating this same block over and over. The block does not
change. Only the count changes.

Each block has two parts. First attention lets every word talk to
every other word. Then a feed forward network lets each word
think privately about what it just heard. Between both parts we
normalize the input and we add a skip connection that lets the
original signal pass through untouched.

The skip connection is the secret to deep networks. Without it
gradients vanish and training fails beyond a few layers. With it
gradients have a highway straight back to the first layer. This
is why we can stack 12 or 96 or 100 of these blocks and they all
keep learning.

In this notebook we build the complete transformer block using
RMSNorm for normalization and SwiGLU for the feed forward layer.
These are the modern choices used by LLaMA and Mistral. We will
also bring in the attention module we wrote in the last chapter
to keep this notebook self contained.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Rotary Position Embeddings

From chapter 4. We need this for the attention module.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## Causal Mask

In [ ]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

## Multi-Head Attention

From chapter 5. The full attention module with RoPE and causal masking.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

## RMSNorm

Root Mean Square normalization. Simpler and faster than LayerNorm.
Divides by the root mean square of the vector. No centering needed.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight

## SwiGLU

A gated feed forward network. Better than ReLU or GELU at scale.
One path produces values. The other path produces gates that
control how much of each value passes through.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

## Transformer Block

This assembles everything. Two sub layers each with pre norm
and a residual connection. Attention mixes information between
tokens. SwiGLU processes each token independently.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## Test the block

Create one transformer block at GPT-2 scale and run it.

In [ ]:
d_model = 768
num_heads = 12
seq_len = 64
batch_size = 4

block = TransformerBlock(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len, x.device)

output = block(x, mask)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Shapes match: {x.shape == output.shape}")
print()

diff = (output - x).abs().mean().item()
print(f"Mean change after block: {diff:.4f}")
print(f"The block changed the input (it did something)")
print()

params = sum(p.numel() for p in block.parameters())
print(f"Parameters per block: {params:,}")
print(f"For a 12 layer GPT-2 Small: {params * 12:,} params in all blocks")

## The residual connection at work

Set attention and FFN to output zero. The input should pass
through unchanged because of the skip connection.

In [ ]:
block.eval()
with torch.no_grad():
    for p in block.attention.parameters():
        p.zero_()
    for p in block.ffn.parameters():
        p.zero_()

x = torch.randn(1, 4, d_model)
mask = create_causal_mask(4, x.device)
output = block(x, mask)

diff = (output - x).abs().max().item()
print(f"Max difference when weights are zero: {diff:.10f}")
print(f"Input passes through unchanged (the residual works)")